In [1]:
import pandas as pd, json, os
df = pd.read_csv("../data/raw/netflix_titles.csv", dtype=str)
rows_before = len(df)
print(df.shape)   # expect (8807, 12)

(8807, 12)


In [2]:
for col in df.columns:
    df[col] = df[col].str.strip().replace("", pd.NA)
print("done; nulls per column:")
print(df.isna().sum().to_string())

done; nulls per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0


In [3]:
mask = df["rating"].str.contains("min", na=False)
df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = pd.NA
df["rating"] = df["rating"].replace({"NR": "Not Rated", "UR": "Not Rated"}).fillna("Unknown")

print("moved:", mask.sum())                              # expect 3
print(df["rating"].value_counts().tail(6))
print("duration nulls:", df["duration"].isna().sum())    # expect 0

moved: 3
rating
TV-G         220
Not Rated     83
G             41
Unknown        7
TV-Y7-FV       6
NC-17          3
Name: count, dtype: int64
duration nulls: 0


In [8]:
# Parse duration into duration_min (movies) and seasons (TV) =====
parts = df["duration"].str.extract(r"^(\d+)\s*(min|Seasons?)$")
num = pd.to_numeric(parts[0]).astype("Int64")
df["duration_min"] = num.where(parts[1] == "min")
df["seasons"] = num.where(parts[1].isin(["Season", "Seasons"]))

print("unparsed duration:        ", int((df["duration"].notna() & parts[0].isna()).sum()))
print("Movies without minutes:   ", int(((df["type"]=="Movie")   & df["duration_min"].isna()).sum()))
print("TV shows without seasons: ", int(((df["type"]=="TV Show") & df["seasons"].isna()).sum()))
print("Movies with seasons:      ", int(((df["type"]=="Movie")   & df["seasons"].notna()).sum()))
print("TV shows with minutes:    ", int(((df["type"]=="TV Show") & df["duration_min"].notna()).sum()))

unparsed duration:         0
Movies without minutes:    0
TV shows without seasons:  0
Movies with seasons:       0
TV shows with minutes:     0


In [7]:
# Group maturity ratings into rating_group =====
RATING_GROUP = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Teen/Family", "PG": "Teen/Family", "TV-14": "Teen/Family", "PG-13": "Teen/Family",
    "TV-MA": "Adult", "R": "Adult", "NC-17": "Adult",
    "Not Rated": "Not Rated / Unknown", "Unknown": "Not Rated / Unknown",
}
df["rating_group"] = df["rating"].map(RATING_GROUP)
print("unmapped:", int(df["rating_group"].isna().sum()))
print(df["rating_group"].value_counts())

unmapped: 0
rating_group
Adult                  4009
Teen/Family            3800
Kids                    908
Not Rated / Unknown      90
Name: count, dtype: int64


In [6]:
# Parse dates, add year_added and added_before_release flag =====
df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y")
df["year_added"] = df["date_added"].dt.year.astype("Int64")
df["release_year"] = df["release_year"].astype(int)
df["added_before_release"] = (df["year_added"] < df["release_year"]).fillna(False)

print("date_added nulls:", int(df["date_added"].isna().sum()))                    # 10
print("year_added range:", df["year_added"].min(), "-", df["year_added"].max())   # 2008 - 2021
print("added before release:", int(df["added_before_release"].sum()))             # 14

date_added nulls: 10
year_added range: 2008 - 2021
added before release: 14


In [9]:
#  Split country and genre into lists, build bridge tables =====
COUNTRY_FIX = {}   # fill in after reviewing the distinct-country list, e.g. {"Old Name": "New Name"}

def split_clean(s, fix=None):
    if pd.isna(s):
        return []
    items = [x.strip() for x in str(s).split(",") if x.strip()]   # strip + drop empty items
    if fix:
        items = [fix.get(x, x) for x in items]
    return list(dict.fromkeys(items))                              # de-dupe, keep order

df["country_list"] = df["country"].map(lambda s: split_clean(s, COUNTRY_FIX))
df["country_known"] = df["country_list"].map(len) > 0
df["country_list"] = df["country_list"].map(lambda L: L if L else ["Unknown"])
df["genre_list"] = df["listed_in"].map(split_clean)

title_country = df[["show_id", "country_list"]].explode("country_list").rename(columns={"country_list": "country"})
title_genre = df[["show_id", "genre_list"]].explode("genre_list").rename(columns={"genre_list": "genre"})

print("titles with unknown country:", int((~df["country_known"]).sum()))   # expect 831
print("distinct countries:", title_country["country"].nunique())            # expect 123 (122 + Unknown)
print("distinct genres:   ", title_genre["genre"].nunique())                # expect 42
print("Indonesia titles:  ", int((title_country["country"] == "Indonesia").sum()))  # expect 90
print("bridge rows:", len(title_country), "country,", len(title_genre), "genre")

titles with unknown country: 831
distinct countries: 123
distinct genres:    42
Indonesia titles:   90
bridge rows: 10843 country, 19323 genre


In [11]:
# List distinct countries (to find naming variants for COUNTRY_FIX) =====
print(sorted(title_country["country"].unique()))

['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bangladesh', 'Belarus', 'Belgium', 'Bermuda', 'Botswana', 'Brazil', 'Bulgaria', 'Burkina Faso', 'Cambodia', 'Cameroon', 'Canada', 'Cayman Islands', 'Chile', 'China', 'Colombia', 'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark', 'Dominican Republic', 'East Germany', 'Ecuador', 'Egypt', 'Ethiopia', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Guatemala', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya', 'Kuwait', 'Latvia', 'Lebanon', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malawi', 'Malaysia', 'Malta', 'Mauritius', 'Mexico', 'Mongolia', 'Montenegro', 'Morocco', 'Mozambique', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Nicaragua', 'Nigeria', 'Norway', 'Pakistan', 'Palestine', 'Panama', 'Paraguay', 'Peru', 'Philippines', 'Poland

In [12]:
# Save clean titles and bridge tables to data/clean/ =====
os.makedirs("../data/clean", exist_ok=True)
titles = df.drop(columns=["country_list", "genre_list"])

assert len(titles) == rows_before, "row count changed"
titles.to_csv("../data/clean/netflix_titles_clean.csv", index=False, encoding="utf-8")
title_country.to_csv("../data/clean/title_country.csv", index=False, encoding="utf-8")
title_genre.to_csv("../data/clean/title_genre.csv", index=False, encoding="utf-8")
print("saved:", len(titles), "titles |", len(title_country), "title-country |", len(title_genre), "title-genre")

saved: 8807 titles | 10843 title-country | 19323 title-genre


In [14]:
# Build profile_after.json and print before/after comparison =====
text_cols = [c for c in titles.columns if pd.api.types.is_string_dtype(titles[c])]
whitespace_rows = sum(int((titles[c].dropna() != titles[c].dropna().str.strip()).sum()) for c in text_cols)

after = {
    "rows": len(titles),
    "dup_show_id": int(titles["show_id"].duplicated().sum()),
    "rating_nunique": int(titles["rating"].nunique()),
    "rating_null": int(titles["rating"].isna().sum()),
    "duration_null": int(titles["duration"].isna().sum()),
    "date_added_null": int(titles["date_added"].isna().sum()),
    "text_cells_with_stray_whitespace": whitespace_rows,
    "country_distinct_clean": int(title_country["country"].nunique()),
    "listed_in_distinct_clean": int(title_genre["genre"].nunique()),
    "country_empty_items": int((title_country["country"].str.strip() == "").sum()),
    "country_unknown_titles": int((~titles["country_known"]).sum()),
    "indonesia_titles": int((title_country["country"] == "Indonesia").sum()),
}
with open("../docs/profile_after.json", "w") as f:
    json.dump(after, f, indent=2)

b = json.load(open("../docs/profile_before.json"))
before = {
    "rows": b["rows"],
    "dup_show_id": b["dup_show_id"],
    "rating_nunique": b["rating_nunique_raw"],
    "rating_null": b["rating_null"],
    "duration_null": b["duration_null"],
    "date_added_null": b["nulls"]["date_added"],
    "text_cells_with_stray_whitespace": 89,   # 88 date_added + 1 title, from profiling Step 7
    "country_distinct_clean": b["country_distinct_clean"],
    "listed_in_distinct_clean": b["listed_in_distinct_clean"],
    "country_empty_items": b["country_empty_items"],
    "country_unknown_titles": b["nulls"]["country"],
    "indonesia_titles": b["indonesia_titles"],
}
for k, v in after.items():
    print(f"{k:35} before={before[k]!s:8} after={v}")

rows                                before=8807     after=8807
dup_show_id                         before=0        after=0
rating_nunique                      before=17       after=14
rating_null                         before=4        after=0
duration_null                       before=3        after=0
date_added_null                     before=10       after=10
text_cells_with_stray_whitespace    before=89       after=0
country_distinct_clean              before=122      after=123
listed_in_distinct_clean            before=42       after=42
country_empty_items                 before=7        after=0
country_unknown_titles              before=831      after=831
indonesia_titles                    before=90       after=90
